# NeuroMorf: Hermes и открытый компонент Instinct

Конечная проверка в бесплатной CPU-сессии Colab, без API-ключей, Drive mount и обращений к внешним моделям. Закреплённый source commit: `9dd335856dc130842df31047a0af940ae57b1fcc`. Эта же версия прошла GitHub Actions: Hermes SDK fixture и Instinct LocalEngine probe. Сам этот notebook ещё не запускался в Colab.

Реальные исследования выполняет [облачный continuous workflow](https://github.com/Petr111111110000568/neuromorph-agent-os/actions/workflows/continuous.yml) с общим durable лимитом. Здесь нет отдельной кнопки live или второго счётчика. Повторный запуск не очищает каталоги.

Instinct LocalEngine — открытая эвристическая библиотека, связь с облачным instinct.com не установлена; это не дополнительная LLM. [Сравнение](https://github.com/Petr111111110000568/neuromorph-agent-os/blob/main/docs/INSTINCT_COMPARISON_RU.md).


In [ ]:
import os, sys, json, pathlib, subprocess
assert sys.platform == "linux" and os.environ.get("COLAB_RELEASE_TAG"), "Colab CPU session required"
SOURCE_COMMIT = "9dd335856dc130842df31047a0af940ae57b1fcc"
BASE = pathlib.Path("/content/neuromorph-hermes-" + SOURCE_COMMIT[:12])
BASE.mkdir(exist_ok=True)
REPO = BASE / "repo"
env = {"PATH": "/usr/local/bin:/usr/bin:/bin", "LANG": "C.UTF-8", "HOME": str(BASE),
       "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_NOSYSTEM": "1", "GIT_CONFIG_GLOBAL": "/dev/null",
       "COLAB_RELEASE_TAG": "cloud-notebook"}
def checked(argv, cwd=BASE, timeout=1000):
    result = subprocess.run(argv, cwd=cwd, env=env, capture_output=True, text=True, timeout=timeout)
    if result.returncode:
        print(result.stdout[-12000:], result.stderr[-4000:])
    result.check_returncode()
    return result.stdout.strip()
if not (REPO / ".git").exists():
    REPO.mkdir(exist_ok=True)
    checked(["git", "init", "."], REPO)
    checked(["git", "fetch", "--depth=1", "https://github.com/Petr111111110000568/neuromorph-agent-os.git", SOURCE_COMMIT], REPO)
    checked(["git", "checkout", "--detach", "FETCH_HEAD"], REPO)
assert checked(["git", "rev-parse", "HEAD"], REPO) == SOURCE_COMMIT
assert not checked(["git", "status", "--porcelain", "--untracked-files=no"], REPO)
print("Pinned project:", SOURCE_COMMIT)
checked([sys.executable, "scripts/bootstrap_hermes.py", "--execute", "--output-dir", "runtime/hermes-harness"], REPO)
print(checked([sys.executable, "-m", "workbench.harnesses.hermes_qwen"], REPO, 180))
print(checked([sys.executable, "scripts/instinct_local_probe.py", "--execute", "--output-dir", "runtime/instinct-probe"], REPO, 90))
for name in ("runtime/hermes-harness/receipt.json", "runtime/hermes-fixture/receipt.json", "runtime/instinct-probe/receipt.json"):
    print(name, json.dumps(json.loads((REPO / name).read_text()), ensure_ascii=False, indent=2))

